# 02 · Instruction tuning: chat templates and response-only loss

*Large Language Model Training*, notebook 2 of 4 · lesson 3: **Prepare supervised instruction tuning**

A pretrained model continues text; it does not answer requests. **Supervised fine-tuning** (SFT) teaches it to answer by training on conversations written in a fixed format. Two details decide whether that works: the **chat template**, the exact token sequence that marks who is speaking, and the **loss mask**, which decides which tokens the model is trained to produce. In this notebook you build both, fine-tune a tiny pretrained model on four instructions over three-letter strings, and measure what changes when the prompt is scored as well as the response. The instructions are symbol games, so the results show the mechanics of instruction tuning, not language understanding.

## What you will build

- `render_chat(messages, add_generation_prompt)`: the chat template, from a list of messages to one string;
- `assistant_flags(messages)`: one flag per token, marking the tokens the assistant should learn to produce;
- `response_only_targets(ids, flags)`: inputs, next-token targets and a loss mask that follows the targets;
- two fine-tuning runs from the same pretrained base, one scoring only the responses and one scoring every token, compared on 40 held-out strings.

## Learning objectives

By the end of this notebook you will be able to:

1. serialise a conversation with a chat template, and use the same template for training and inference;
2. build a response-only loss mask aligned with the shifted next-token targets, and test it with an invariant;
3. handle padding and truncation so that every training example keeps at least one supervised token;
4. measure the difference between scoring only the responses and scoring every token, and explain it.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for one plot. Nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded: the data are generated in the notebook from fixed seeds, so your numbers should match the ones quoted in the text up to small differences between computers.
- Two hidden recap cells hold the tiny tensor engine from Base Model Training and the pieces from notebook 01: the tokeniser, the toy-world corpus, the tiny transformer, Adam and the pretraining recipe.
- The notebook runs in a few seconds on a laptop, most of it in the three training runs; in the browser it takes longer.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, pasted unchanged: `Tensor` (with `+`, `-`, `*`, `/`, `**`, `@`, `exp`, `log`, `tanh`, `relu`, `sigmoid`, `sum`, `mean`, `reshape`, `transpose`, indexing, `log_softmax`, `softmax` and `backward`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it was checked against numerical gradients. Run the cell; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The second hidden cell re-defines what this notebook needs from notebook 01, under the same names:

- the tokeniser: `VOCAB` (46 tokens: five special tokens, then one per character), `encode(text)`, `decode(ids)` and the ids `PAD`, `EOS`, `USER`, `ASSISTANT` and `END`;
- the toy world: `EVAL_STRINGS`, the 40 reserved three-letter strings of the evaluation set; `RESERVED`, those strings and their reversals, which no training text may contain; `random_string(picker, banned)`; and `make_clean_corpus()`;
- the model: `init_tiny_gpt(seed)`, `gpt_forward(params, ids)` (ids `(batch, T)` → logits `(batch, T, 46)`) and `Adam(tensors, lr)`;
- `pretrain_base(steps, seed)`: notebook 01's pretraining recipe on the clean corpus, here without a validation split.

In [ ]:
# 📦 Recap: the pieces from notebook 01, under their usual names.

# ── Vocabulary and tokeniser (notebook 01) ─────────────────────────────────
# One token per character, plus five special tokens. The chat template's
# tokens are reserved from the start, as real tokenisers reserve theirs.
SPECIALS = ["<pad>", "<eos>", "<|user|>", "<|assistant|>", "<|end|>"]
VOCAB = SPECIALS + list(" .,:?0123456789abcdefghijklmnopqrstuvwxyz")
TOKEN_ID = {token: index for index, token in enumerate(VOCAB)}
PAD, EOS, USER, ASSISTANT, END = range(len(SPECIALS))


def encode(text):
    """Text -> list of token ids. Each special token, such as <|end|>, becomes a single id."""
    ids, i = [], 0
    while i < len(text):
        special = next((s for s in SPECIALS if text.startswith(s, i)), None)
        if special is not None:
            ids.append(TOKEN_ID[special])
            i += len(special)
        else:
            ids.append(TOKEN_ID[text[i]])
            i += 1
    return ids


def decode(ids):
    """List of token ids -> text."""
    return "".join(VOCAB[int(i)] for i in ids)


# ── The toy world: three sources of generated text (notebook 01) ────────────
LETTERS = "abcdefghijklmnopqrstuvwxyz"
ANIMALS = ["fox", "owl", "cat", "hen", "elk", "bee", "yak", "eel", "ram", "doe"]
COLOURS = ["red", "tan", "grey", "brown", "white", "black", "gold"]
VERBS = ["ran", "sat", "slept", "hid", "sang", "swam", "ate"]
PREPOSITIONS = ["to", "by", "near", "in"]
PLACES = ["river", "forest", "hill", "barn", "pond", "field", "lake"]
ITEMS = ["figs", "plums", "nuts", "pears", "eggs", "beans", "limes"]
NUMBERS = ["one", "two", "three", "four", "five", "six"]


def evaluation_strings(count=40, seed=123):
    """The reserved three-letter strings of the held-out evaluation set, sorted."""
    picker = np.random.default_rng(seed)
    chosen = set()
    while len(chosen) < count:
        chosen.add("".join(picker.choice(list(LETTERS), 3)))
    return sorted(chosen)


EVAL_STRINGS = evaluation_strings()
RESERVED = set(EVAL_STRINGS) | {s[::-1] for s in EVAL_STRINGS}  # no training text may contain these


def random_string(picker, banned=()):
    """A random three-letter string that is not in `banned`."""
    while True:
        s = "".join(picker.choice(list(LETTERS), 3))
        if s not in banned:
            return s


def make_document(source, picker, banned=()):
    """One generated document from 'stories', 'letters' or 'lists'."""
    if source == "stories":
        return " ".join(
            f"the {picker.choice(COLOURS)} {picker.choice(ANIMALS)} {picker.choice(VERBS)} "
            f"{picker.choice(PREPOSITIONS)} the {picker.choice(PLACES)}."
            for _ in range(picker.integers(2, 5))
        )
    if source == "letters":
        sentences = []
        for _ in range(picker.integers(2, 5)):
            s = random_string(picker, banned)
            sentences.append([f"{s} backwards is {s[::-1]}.", f"{s} starts with {s[0]}.",
                              f"{s} ends with {s[-1]}.", f"say {s} twice: {s} {s}."][picker.integers(4)])
        return " ".join(sentences)
    picks = picker.choice(ITEMS, picker.integers(2, 4), replace=False)
    return "list: " + ", ".join(f"{picker.choice(NUMBERS)} {item}" for item in picks) + "."


def make_clean_corpus(n_docs=500, seed=7):
    """Generated documents with ids, sources and text. Letter games never use a reserved string."""
    picker = np.random.default_rng(seed)
    docs = []
    for i in range(n_docs):
        source = str(picker.choice(["stories", "letters", "lists"], p=[0.45, 0.4, 0.15]))
        docs.append({"doc_id": f"d{i:04d}", "source": source,
                     "text": make_document(source, picker, RESERVED)})
    return docs


def project(x, params, name, adapters=None):
    """x @ W for the named weight matrix. (Notebook 03 attaches low-rank adapters here.)"""
    return x @ params[name]


# ── A tiny causal transformer, written with the tensor engine ───────────────
# ids (batch, T) -> embeddings (batch, T, D) -> logits (batch, T, len(VOCAB)).
# Each block is pre-norm: x + attention(layer_norm(x)), then x + mlp(layer_norm(x)).
CONTEXT, D_MODEL, N_HEADS, N_LAYERS, D_FF = 32, 32, 2, 2, 128


def init_tiny_gpt(seed=0):
    """Every weight of the model, as a dict name -> trainable Tensor."""
    init = np.random.default_rng(seed)
    D = D_MODEL
    params = {"tok": parameter(len(VOCAB), D, rng=init, scale=0.1),
              "pos": parameter(CONTEXT, D, rng=init, scale=0.1)}
    for b in range(N_LAYERS):
        params[f"b{b}.ln1_g"], params[f"b{b}.ln1_b"] = Tensor(np.ones(D), requires_grad=True), zeros_parameter(D)
        for name in ("Wq", "Wk", "Wv"):
            params[f"b{b}.{name}"] = parameter(D, D, rng=init, scale=D ** -0.5)
        params[f"b{b}.Wo"] = parameter(D, D, rng=init, scale=(2 * N_LAYERS * D) ** -0.5)
        params[f"b{b}.ln2_g"], params[f"b{b}.ln2_b"] = Tensor(np.ones(D), requires_grad=True), zeros_parameter(D)
        params[f"b{b}.W1"], params[f"b{b}.b1"] = parameter(D, D_FF, rng=init, scale=D ** -0.5), zeros_parameter(D_FF)
        params[f"b{b}.W2"] = parameter(D_FF, D, rng=init, scale=(2 * N_LAYERS * D_FF) ** -0.5)
        params[f"b{b}.b2"] = zeros_parameter(D)
    params["lnf_g"], params["lnf_b"] = Tensor(np.ones(D), requires_grad=True), zeros_parameter(D)
    params["Wout"] = parameter(D, len(VOCAB), rng=init, scale=D ** -0.5)
    return params


def gpt_forward(params, ids, adapters=None):
    """Next-token logits (batch, T, len(VOCAB)) for integer token ids (batch, T), with T <= CONTEXT."""
    ids = np.asarray(ids)
    batch, T = ids.shape
    D = params["tok"].shape[1]
    hd = D // N_HEADS
    future = np.triu(np.full((T, T), -1e9), 1)  # position t may not look at positions after t
    x = params["tok"][ids] + params["pos"][np.arange(T)]
    for b in range(N_LAYERS):
        h = layer_norm(x, params[f"b{b}.ln1_g"], params[f"b{b}.ln1_b"])
        q, k, v = (project(h, params, f"b{b}.{name}", adapters).reshape(batch, T, N_HEADS, hd).transpose(0, 2, 1, 3)
                   for name in ("Wq", "Wk", "Wv"))
        attention = ((q @ k.transpose(0, 1, 3, 2)) * hd ** -0.5 + future).softmax(axis=-1)
        mixed = (attention @ v).transpose(0, 2, 1, 3).reshape(batch, T, D)
        x = x + project(mixed, params, f"b{b}.Wo", adapters)
        h = layer_norm(x, params[f"b{b}.ln2_g"], params[f"b{b}.ln2_b"])
        hidden = (project(h, params, f"b{b}.W1", adapters) + params[f"b{b}.b1"]).relu()
        x = x + project(hidden, params, f"b{b}.W2", adapters) + params[f"b{b}.b2"]
    return project(layer_norm(x, params["lnf_g"], params["lnf_b"]), params, "Wout", adapters)


class Adam:
    """Adam with bias correction and clipping of the global gradient norm, for a list of Tensors."""

    def __init__(self, tensors, lr=3e-3, betas=(0.9, 0.99), eps=1e-8, clip=1.0):
        self.tensors, self.lr, self.betas, self.eps, self.clip = list(tensors), lr, betas, eps, clip
        self.m = [np.zeros_like(t.data) for t in self.tensors]
        self.v = [np.zeros_like(t.data) for t in self.tensors]
        self.t = 0

    def step(self):
        self.t += 1
        b1, b2 = self.betas
        grads = [np.zeros_like(t.data) if t.grad is None else t.grad for t in self.tensors]
        norm = float(np.sqrt(sum((g ** 2).sum() for g in grads)))
        shrink = min(1.0, self.clip / (norm + 1e-12))
        for t, g, m, v in zip(self.tensors, grads, self.m, self.v):
            g = g * shrink
            m[...] = b1 * m + (1 - b1) * g
            v[...] = b2 * v + (1 - b2) * g * g
            t.data -= self.lr * (m / (1 - b1 ** self.t)) / (np.sqrt(v / (1 - b2 ** self.t)) + self.eps)
        return norm


# ── Pretraining (notebook 01) ───────────────────────────────────────────────
def pack_documents(docs):
    """One token stream: every document followed by <eos>."""
    stream = []
    for doc in docs:
        stream += encode(doc["text"]) + [EOS]
    return np.array(stream)


def pretrain_base(steps=150, seed=0, batch_size=16, lr=3e-3):
    """A shorter run of notebook 01's recipe: random windows of the packed clean corpus."""
    stream = pack_documents(make_clean_corpus())
    params = init_tiny_gpt(seed)
    optimiser = Adam(list(params.values()), lr=lr)
    window_picker = np.random.default_rng(seed)
    for _ in range(steps):
        starts = window_picker.integers(0, len(stream) - CONTEXT - 1, batch_size)
        windows = np.stack([stream[s:s + CONTEXT + 1] for s in starts])
        loss = cross_entropy(gpt_forward(params, windows[:, :-1]), windows[:, 1:])
        zero_grad(optimiser.tensors)
        loss.backward()
        optimiser.step()
    return params

## 1 · Four instructions and a pretrained base

The instructions are four small games on a three-letter string $s$: `copy` returns $s$, `reverse` returns it backwards, `first` returns its first letter and `last` its last letter. Training conversations use random strings; the 40 reserved strings, and their reversals, never appear in training, so the evaluation measures whether the model learned the *rule* rather than memorised answers.

First pretrain a base model with a shorter version of notebook 01's recipe (150 steps), and look at what it does with an instruction. `continue_text` simply appends the most likely next token, again and again.

In [ ]:
TASKS = {"copy": lambda s: s, "reverse": lambda s: s[::-1], "first": lambda s: s[0], "last": lambda s: s[-1]}
for task, rule in TASKS.items():
    print(f"{task} abc -> {rule('abc')}")


def continue_text(params, text, n_tokens):
    """Greedy continuation: append the most likely next token n_tokens times, ignoring <|end|>."""
    ids = encode(text)
    for _ in range(n_tokens):
        ids.append(int(gpt_forward(params, np.array([ids[-CONTEXT:]])).data[0, -1].argmax()))
    return decode(ids)


start = time.perf_counter()
base = pretrain_base(steps=150, seed=0)
print(f"\npretrained the base model in {time.perf_counter() - start:.1f} s")
print(repr(continue_text(base, "copy abc ", 20)))

**What to notice.** The base model continues the text with corpus-like fragments. It learned the statistics of stories, letter games and lists; nothing in pretraining showed it a request followed by an answer.

## 2 · The chat template

A conversation is a list of messages, each with a `role` and some `content`. The **chat template** turns it into one string, and so into one token sequence. Ours wraps every message as

$$\texttt{<|role|>}\ \text{content}\ \texttt{<|end|>},$$

so *copy abc* answered by *abc* becomes `<|user|>copy abc<|end|><|assistant|>abc<|end|>`. Each marker is a single special token, reserved in the vocabulary since notebook 01.

At inference time the template is rendered **without** the answer and ends with the opening marker `<|assistant|>`, the *generation prompt*, so that the model's next token begins the reply; it then writes until it produces `<|end|>`. The template must be identical in training and inference. A missing generation prompt, a duplicated special token or a different separator changes the token sequence the model sees and can undo the effect of otherwise good data. Real tokenisers store their template next to the vocabulary (Hugging Face calls it a chat template), precisely so that both sides use the same one.

In [ ]:
text = "<|user|>copy abc<|end|><|assistant|>abc<|end|>"
ids = encode(text)
print(len(text), "characters ->", len(ids), "tokens:", ids)
print([VOCAB[i] for i in ids])

### ✍️ Your turn: `render_chat`

Write `render_chat(messages, add_generation_prompt=False)`.

- Input: `messages`, a list of dicts with keys `"role"` (`"user"` or `"assistant"`) and `"content"` (a string); and a flag.
- Output: one string, the concatenation of `f"<|{role}|>{content}<|end|>"` for every message in order, followed by `"<|assistant|>"` when `add_generation_prompt` is true.

Hint: `"".join(...)` over a generator expression builds the string in one line.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below; open it only when you are stuck.

In [ ]:
def render_chat_reference(messages, add_generation_prompt=False):
    text = "".join(f"<|{m['role']}|>{m['content']}<|end|>" for m in messages)
    return text + ("<|assistant|>" if add_generation_prompt else "")

In [ ]:
def render_chat(messages, add_generation_prompt=False):
    """<|role|>content<|end|> for every message, then <|assistant|> if add_generation_prompt is true."""
    # ✍️ Your code here (about two lines)
    raise NotImplementedError

In [ ]:
one_turn = [{"role": "user", "content": "copy abc"}, {"role": "assistant", "content": "abc"}]
two_turns = one_turn + [{"role": "user", "content": "last xyz"}, {"role": "assistant", "content": "z"}]
render_chat = practice("render_chat", render_chat, render_chat_reference,
                       cases=[(one_turn,), (two_turns,), ([{"role": "user", "content": "first xyz"}], True), ([], True)])
print(render_chat(two_turns))

**Expected behaviour.** The one-turn case gives exactly the string in the demo, the two-turn case continues with `<|user|>last xyz<|end|><|assistant|>z<|end|>`, and a generation prompt ends the string with `<|assistant|>` and no `<|end|>` after it. An empty conversation with a generation prompt is just `<|assistant|>`.

We use one-turn conversations from here on. `task_messages` builds them from a task name and a string.

In [ ]:
def task_messages(task, s, with_answer=True):
    """A one-turn conversation asking for `task` on the string s."""
    messages = [{"role": "user", "content": f"{task} {s}"}]
    if with_answer:
        messages.append({"role": "assistant", "content": TASKS[task](s)})
    return messages


print(render_chat(task_messages("reverse", "abc")))
print(render_chat(task_messages("reverse", "abc", with_answer=False), add_generation_prompt=True))

## 3 · Which tokens should the loss score?

SFT should teach the model to produce the assistant's replies, **including** the closing `<|end|>`, because that token tells the inference loop to stop. The user's text is context: the model must read it, but it is not asked to write it. The role markers are written by the template, not by the model. So we give every token of `encode(render_chat(messages))` a **flag**: 1 for the tokens of an assistant message's content and for the `<|end|>` that closes it, and 0 for everything else.

For *copy abc*:

| token | `<|user|>` | c | o | p | y | ␣ | a | b | c | `<|end|>` | `<|assistant|>` | a | b | c | `<|end|>` |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| flag | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 1 | 1 | 1 | 1 |

In a conversation with several assistant turns, flagging every turn is a decision, not a law; some recipes score only the last reply (Exercise 1). Libraries expose the choice explicitly, for example as an "assistant-only" or "completion-only" loss option.

In [ ]:
for part in ["<|user|>", "copy abc", "<|end|>", "<|assistant|>", "abc", "<|end|>"]:
    print(f"{part!r:17} -> {len(encode(part))} token(s)")

### ✍️ Your turn: `assistant_flags`

Write `assistant_flags(messages)`, which returns a list of 0s and 1s with one entry per token of `encode(render_chat(messages))`.

- For each message: one flag for its role marker (always 0), one flag per token of its content, and one flag for its closing `<|end|>`.
- The content and closing flags are 1 for an assistant message and 0 for a user message.

Hint: build the list message by message, using `len(encode(m["content"]))` for the number of content tokens, as the demo above does piece by piece.

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def assistant_flags_reference(messages):
    flags = []
    for m in messages:
        scored = 1 if m["role"] == "assistant" else 0
        flags += [0] + [scored] * len(encode(m["content"])) + [scored]
    return flags

In [ ]:
def assistant_flags(messages):
    """One 0/1 flag per token of encode(render_chat(messages)): 1 for assistant content and its <|end|>."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
empty_reply = [{"role": "user", "content": "copy abc"}, {"role": "assistant", "content": ""}]
assistant_flags = practice("assistant_flags", assistant_flags, assistant_flags_reference,
                           cases=[(one_turn,), (two_turns,), (empty_reply,)])
ids = encode(render_chat(two_turns))
flags = assistant_flags(two_turns)
assert len(ids) == len(flags)
print(" ".join(f"{VOCAB[i]}:{f}" for i, f in zip(ids, flags)))

**Expected behaviour.** The one-turn case reproduces the table above. In the two-turn case both replies, *abc* and *z*, are flagged together with their `<|end|>`s. An empty reply flags only its `<|end|>`: the model is still taught to stop.

## 4 · Shift once, and let the mask follow the targets

A causal language model at position $t$ predicts the token at position $t + 1$. For a sequence of $L$ tokens, the inputs are tokens $0 \dots L-2$ and the targets are tokens $1 \dots L-1$:

$$\text{inputs} = \text{ids}[{:}{-1}], \qquad \text{targets} = \text{ids}[1{:}], \qquad \text{mask} = \text{flags}[1{:}].$$

The mask describes the **targets**, so it is shifted with them. Mask position $t$ is 1 when the token the model must predict there is part of a reply. With the loss

$$\mathcal{L} = -\frac{\sum_t m_t \log p_\theta(y_t \mid x_{\le t})}{\sum_t m_t},$$

masked positions contribute nothing, and the average is over supervised tokens only; the engine's `cross_entropy(logits, targets, mask)` computes exactly this. Note that the prompt stays in the **inputs**: the model reads it at every position. Context and supervision are separate decisions.

A common bug is to shift the ids but not the flags, or to shift the flags the wrong way. The demo lines the three rows up for *copy abc*.

In [ ]:
ids = encode(render_chat(one_turn))
flags = assistant_flags(one_turn)
print("position:", " ".join(f"{t:>3}" for t in range(len(ids) - 1)))
print("input:   ", " ".join(f"{VOCAB[i][:3]:>3}" for i in ids[:-1]))
print("target:  ", " ".join(f"{VOCAB[i][:3]:>3}" for i in ids[1:]))
print("mask:    ", " ".join(f"{f:>3}" for f in flags[1:]))

### ✍️ Your turn: `response_only_targets`

Write `response_only_targets(ids, flags)`, which returns a tuple `(inputs, targets, mask)` of NumPy arrays, each of length `len(ids) - 1`:

- `inputs = ids[:-1]` and `targets = ids[1:]`, as integer arrays;
- `mask = flags[1:]`, as a float array.

Hint: convert first, with `np.asarray(ids)` and `np.asarray(flags, dtype=float)`. Do not change the lists you are given.

*Type it rather than pasting it: typing is how the shapes and indices sink in.*

In [ ]:
def response_only_targets_reference(ids, flags):
    ids, flags = np.asarray(ids), np.asarray(flags, dtype=float)
    return ids[:-1], ids[1:], flags[1:]

In [ ]:
def response_only_targets(ids, flags):
    """(inputs, targets, mask): ids[:-1], ids[1:] as integers and flags[1:] as floats."""
    # ✍️ Your code here (about two lines)
    raise NotImplementedError

In [ ]:
response_only_targets = practice(
    "response_only_targets", response_only_targets, response_only_targets_reference,
    cases=[(encode(render_chat(one_turn)), assistant_flags(one_turn)), ([7, 8], [0, 1]), ([7, 8, 9], [0, 0, 0])])

**Expected behaviour.** For *copy abc*, the mask has 1s at the last four target positions, whose targets are *a*, *b*, *c* and `<|end|>`; the position whose *input* is `<|assistant|>` is the first scored one, because that is where the model must produce the first letter of the reply. A sequence without any flagged token produces an all-zero mask, which is what truncation can do to an example (section 5).

## 5 · Batches: padding and truncation

Conversations have different lengths (*first abc* is shorter than *reverse abc*), so a batch pads the shorter rows with `<pad>` up to the longest one. Padding is added *after* the shift, with mask 0, so padded targets never count. `sft_batch` builds a batch of one-turn conversations on random training strings, which never include a reserved string; the instruction of each row is drawn from `tasks`, optionally with weights.

**Truncation** is the other trap. Long examples are often cut to a maximum number of tokens, and the cut falls at the end, where the reply is. An example whose reply has been cut away has no supervised token left: it costs compute and teaches nothing, and a batch made only of such examples would divide by zero if the loss did not guard against it. Check after truncation, and either drop such examples or shorten the prompt instead.

In [ ]:
def pad_rows(rows):
    """Stack (inputs, targets, mask) rows of different lengths; padding gets <pad> and mask 0."""
    T = max(len(inputs) for inputs, _, _ in rows)
    inputs, targets, mask = np.full((len(rows), T), PAD), np.full((len(rows), T), PAD), np.zeros((len(rows), T))
    for i, (x, y, m) in enumerate(rows):
        inputs[i, :len(x)], targets[i, :len(y)], mask[i, :len(m)] = x, y, m
    return inputs, targets, mask


def sft_batch(tasks, batch_size, seed, task_weights=None):
    """A padded batch of one-turn conversations on training strings (never the evaluation strings)."""
    picker = np.random.default_rng(seed)
    rows = []
    for _ in range(batch_size):
        task = str(picker.choice(tasks, p=task_weights))
        messages = task_messages(task, random_string(picker, RESERVED))
        rows.append(response_only_targets(encode(render_chat(messages)), assistant_flags(messages)))
    return pad_rows(rows)


inputs, targets, mask = sft_batch(list(TASKS), batch_size=3, seed=0)
for row in range(3):
    print(repr(decode(inputs[row])), "| supervised targets:", repr(decode(targets[row][mask[row] == 1])))

In [ ]:
max_tokens = 12  # 🧪 try 16 and 18, then run this cell again
messages = task_messages("reverse", "abc")
ids, flags = encode(render_chat(messages))[:max_tokens], assistant_flags(messages)[:max_tokens]
_, kept_targets, kept_mask = response_only_targets(ids, flags)
print(f"{len(encode(render_chat(messages)))} tokens, truncated to {len(ids)}: {int(kept_mask.sum())} supervised targets",
      repr(decode(kept_targets[kept_mask == 1])))
if kept_mask.sum() == 0:
    print("no supervised token is left: drop this example, or shorten the prompt instead of the reply")

**What to notice.** *reverse abc* takes 18 tokens. Truncated to 12, nothing of the reply survives; at 16 the model would be taught *cb* and never shown the third letter or the `<|end|>`, so it would learn to stop in the wrong place, which is worse than learning nothing. Our conversations are short, so no training example here is truncated, and the put-it-together cell checks that every row keeps at least one supervised token.

## 6 · Fine-tune twice: score only the replies, or every token?

`sft_train` runs supervised fine-tuning: each step builds a fresh batch of 32 conversations, computes the masked cross-entropy, back-propagates and updates every weight with Adam. With `score_prompts=True` it replaces the mask by "every target that is not padding", which is what you get from a plain language-modelling collator or a forgotten mask.

Both runs start from copies of the same base, see exactly the same 250 batches, and use the same learning rate. Only the mask differs.

In [ ]:
def sft_train(params, tasks, steps, seed=0, lr=3e-3, batch_size=32, task_weights=None,
              score_prompts=False, adapters=None):
    """Supervised fine-tuning; returns the loss of every step. With adapters, only A and B are trained."""
    tensors = [t for A, B, _ in adapters.values() for t in (A, B)] if adapters else list(params.values())
    optimiser = Adam(tensors, lr=lr)
    losses = []
    for step in range(steps):
        inputs, targets, mask = sft_batch(tasks, batch_size, seed * 100_000 + step, task_weights)
        if score_prompts:  # the variant without the response-only mask
            mask = (targets != PAD).astype(float)
        loss = cross_entropy(gpt_forward(params, inputs, adapters), targets, mask)
        zero_grad(tensors)
        loss.backward()
        optimiser.step()
        losses.append(float(loss.data))
    return losses


response_only = {name: Tensor(t.data.copy(), requires_grad=True) for name, t in base.items()}
every_token = {name: Tensor(t.data.copy(), requires_grad=True) for name, t in base.items()}
start = time.perf_counter()
losses_response_only = sft_train(response_only, list(TASKS), steps=250, seed=0)
losses_every_token = sft_train(every_token, list(TASKS), steps=250, seed=0, score_prompts=True)
print(f"two fine-tuning runs of 250 steps in {time.perf_counter() - start:.1f} s")

To evaluate, render each prompt with the generation prompt, let the model write greedily until it produces `<|end|>`, and compare the reply with the right answer, character for character (exact match). `generate` decodes prompts of the same token length together; `task_accuracy` scores each instruction on the 40 reserved strings, none of which the model saw in training.

In [ ]:
def generate(params, prompts, adapters=None, max_new_tokens=5):
    """Greedy decoding until <|end|>. Prompts of the same token length are decoded together."""
    encoded = [encode(p) for p in prompts]
    answers = [None] * len(prompts)
    for length in sorted({len(ids) for ids in encoded}):
        group = [i for i, ids in enumerate(encoded) if len(ids) == length]
        tokens = np.array([encoded[i] for i in group])
        out, done = [[] for _ in group], np.zeros(len(group), dtype=bool)
        for _ in range(max_new_tokens):
            next_ids = gpt_forward(params, tokens, adapters).data[:, -1].argmax(axis=-1)
            for j, token in enumerate(next_ids):
                if not done[j] and token == END:
                    done[j] = True
                elif not done[j]:
                    out[j].append(token)
            if done.all():
                break
            tokens = np.concatenate([tokens, next_ids[:, None]], axis=1)
        for j, i in enumerate(group):
            answers[i] = decode(out[j])
    return answers


def task_accuracy(params, tasks, strings=EVAL_STRINGS, adapters=None):
    """Exact-match accuracy of greedy answers on the given strings, per task."""
    scores = {}
    for task in tasks:
        prompts = [render_chat(task_messages(task, s, with_answer=False), add_generation_prompt=True) for s in strings]
        answers = generate(params, prompts, adapters)
        scores[task] = float(np.mean([a == TASKS[task](s) for a, s in zip(answers, strings)]))
    return scores


print("accuracy on the 40 reserved strings (exact match)")
for name, params in [("base, no fine-tuning", base), ("replies scored", response_only), ("every token scored", every_token)]:
    print(f"  {name:<21}", {task: round(score, 3) for task, score in task_accuracy(params, list(TASKS)).items()})

Accuracy looks only at the reply. To see what each run learned about the **prompt**, measure the average loss separately at prompt positions (mask 0, not padding) and at reply positions (mask 1) on held-out conversations: a low prompt loss means the model has learned to predict, and so to produce, the user's side of the conversation.

In [ ]:
def position_losses(params, tasks=tuple(TASKS)):
    """Mean next-token loss at prompt positions and at reply positions of the held-out conversations."""
    rows = []
    for task in tasks:
        for s in EVAL_STRINGS:
            messages = task_messages(task, s)
            rows.append(response_only_targets(encode(render_chat(messages)), assistant_flags(messages)))
    inputs, targets, mask = pad_rows(rows)
    log_probs = gpt_forward(params, inputs).log_softmax(axis=-1).data
    token_loss = -np.take_along_axis(log_probs, targets[..., None], axis=-1)[..., 0]
    prompt = ((mask == 0) & (targets != PAD)).astype(float)
    return float((token_loss * prompt).sum() / prompt.sum()), float((token_loss * mask).sum() / mask.sum())


loss_table = {name: position_losses(params) for name, params in
              [("base, no fine-tuning", base), ("replies scored", response_only), ("every token scored", every_token)]}
print("                       prompt loss   reply loss   (nats per token, held-out conversations)")
for name, (prompt_loss, reply_loss) in loss_table.items():
    print(f"  {name:<21} {prompt_loss:>10.3f} {reply_loss:>12.3f}")

**What to notice.**

- The base model fails every instruction: it has never seen this format.
- Scoring only the replies teaches all four rules, measured on strings the model never saw in training. Its prompt loss is *higher* than the base model's: no gradient ever asked it to predict a prompt, while the shared weights changed for the replies.
- Scoring every token also teaches the rules, but in this run it is still behind on `last` after the same 250 steps. Most of its loss comes from prompt tokens, and the three random letters in every prompt cost about $\ln 26 \approx 3.3$ nats each, a floor no model can beat, so a large part of each update is spent on targets it cannot learn. With more steps it would probably catch up; this is one measured run, not a law.
- Its prompt loss is low: it has learned to predict the user's side of the conversation. The experiment below shows what that means.

### 🧪 Experiment: who speaks next?

Start each fine-tuned model with nothing but `<|user|>` and let it continue greedily, ignoring `<|end|>`. Then change the starting text, for example to `"<|user|>rev"`, and run the cell again.

In [ ]:
start_text = "<|user|>"  # 🧪 try "<|user|>rev" or "<|user|>first q"
for name, params in [("replies scored", response_only), ("every token scored", every_token)]:
    print(f"{name:<19}", repr(continue_text(params, start_text, 14)))

**What to notice.** The model trained on every token writes a plausible user request, answers it itself and carries on: it was trained to reproduce prompts, so it does. The model trained on replies only has no idea what a user turn looks like; its continuation is junk. Masking the prompt stops the model being trained to imitate the user.

Scoring prompts is not always wrong: some recipes keep a small weight on prompt tokens, and with very short replies that can help. The point is that it is a choice with measurable consequences, so make it on purpose and check the collated batch. Note too that the two runs' **training losses cannot be compared**: they average over different sets of tokens.

In [ ]:
plt.figure(figsize=(6.5, 3.4))
plt.plot(losses_response_only, label="replies scored (mean over reply tokens)")
plt.plot(losses_every_token, label="every token scored (mean over all tokens)")
plt.yscale("log")
plt.xlabel("step")
plt.ylabel("training loss (nats per token)")
plt.legend()
plt.tight_layout()
plt.show()

## Put it together

The final cell checks the invariants that make response-only SFT correct, and the measured behaviour of the two runs:

1. **The template round-trips**: decoding the encoded template gives the same string back.
2. **A masked target cannot change the loss**: replacing every target at a masked position by random tokens leaves the loss exactly unchanged, and the gradient of the loss with respect to the logits is zero at masked positions.
3. **No empty examples**: every row of every training batch keeps at least one supervised token.
4. **Measured behaviour**: the replies-only run reaches at least 85% exact match on average over the four instructions (it measured 100% here), and the run that scored every token has learned the prompts, with a prompt loss at least 3 nats lower.

In [ ]:
# 1. The template round-trips through the tokeniser.
for messages in (one_turn, two_turns, task_messages("last", "xyz")):
    assert decode(encode(render_chat(messages))) == render_chat(messages)

# 2. Masked targets cannot change the loss, and receive no gradient.
inputs, targets, mask = sft_batch(list(TASKS), batch_size=16, seed=123)
logits = Tensor(gpt_forward(response_only, inputs).data, requires_grad=True)
loss = cross_entropy(logits, targets, mask)
scrambled = np.where(mask == 1, targets, np.random.default_rng(5).integers(0, len(VOCAB), targets.shape))
assert float(cross_entropy(logits, scrambled, mask).data) == float(loss.data), "a masked target changed the loss"
loss.backward()
assert np.all(logits.grad[mask == 0] == 0.0), "a masked position received a gradient"

# 3. Every training row keeps at least one supervised token.
for step in range(5):
    assert (sft_batch(list(TASKS), 32, step)[2].sum(axis=1) > 0).all()

# 4. Measured behaviour on the 40 reserved strings and the held-out conversations.
accuracy = task_accuracy(response_only, list(TASKS))
mean_accuracy = float(np.mean(list(accuracy.values())))
assert mean_accuracy >= 0.85, f"replies-only fine-tuning reached only {mean_accuracy:.0%}"
prompt_loss_masked, prompt_loss_unmasked = loss_table["replies scored"][0], loss_table["every token scored"][0]
assert prompt_loss_unmasked < prompt_loss_masked - 3.0, "the every-token run should have learned the prompts"
print("✅ All checks passed.")
print(f"replies scored: {mean_accuracy:.0%} exact match; prompt loss {prompt_loss_masked:.2f} (replies scored) "
      f"vs {prompt_loss_unmasked:.2f} (every token scored)")

## Exercises

The first two have checks; the last two are questions to answer in a markdown cell of your own.

### Exercise 1 ✍️ `last_turn_flags`: supervise only the final reply

Some recipes train only on the last assistant reply of a multi-turn conversation, treating earlier replies as context. Write `last_turn_flags(messages)`: the same as `assistant_flags`, except that only the **last** assistant message (its content and its `<|end|>`) is flagged.

Hint: find the index of the last assistant message first, then build the flags as before.

In [ ]:
def last_turn_flags_reference(messages):
    last = max((i for i, m in enumerate(messages) if m["role"] == "assistant"), default=-1)
    flags = []
    for i, m in enumerate(messages):
        scored = 1 if i == last else 0
        flags += [0] + [scored] * len(encode(m["content"])) + [scored]
    return flags

In [ ]:
def last_turn_flags(messages):
    """Like assistant_flags, but only the last assistant message (content and <|end|>) is flagged."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
last_turn_flags = practice("last_turn_flags", last_turn_flags, last_turn_flags_reference,
                           cases=[(one_turn,), (two_turns,), ([{"role": "user", "content": "hi"}],)])

### Exercise 2 ✍️ `per_sequence_loss`: two ways to average

`cross_entropy` averages over all supervised tokens of the batch, so a long reply carries more weight than a short one. The alternative first averages within each sequence and then over sequences. Write `per_sequence_loss(token_losses, mask)` for arrays of shape `(batch, T)`: the mean over the rows that have at least one supervised token of each row's masked mean. Then compare the two averages for a batch in which one reply has one token and another has four.

In [ ]:
def per_sequence_loss_reference(token_losses, mask):
    token_losses, mask = np.asarray(token_losses, dtype=float), np.asarray(mask, dtype=float)
    counts = mask.sum(axis=1)
    rows = counts > 0
    return float(np.mean((token_losses * mask).sum(axis=1)[rows] / counts[rows]))

In [ ]:
def per_sequence_loss(token_losses, mask):
    """Mean over rows with a supervised token of (sum of masked losses in the row / supervised tokens in the row)."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
losses_example = np.array([[2.0, 0.0, 0.0, 0.0], [1.0, 1.0, 1.0, 1.0], [5.0, 5.0, 5.0, 5.0]])
mask_example = np.array([[1.0, 0.0, 0.0, 0.0], [1.0, 1.0, 1.0, 1.0], [0.0, 0.0, 0.0, 0.0]])
per_sequence_loss = practice("per_sequence_loss", per_sequence_loss, per_sequence_loss_reference,
                             cases=[(losses_example, mask_example), (np.ones((2, 3)), np.ones((2, 3)))])
print("per token:", (losses_example * mask_example).sum() / mask_example.sum(),
      " per sequence:", per_sequence_loss(losses_example, mask_example))

### Exercise 3 (question): why does the prompt loss rise?

Before fine-tuning, the base model's prompt loss was below 5 nats per token; after replies-only fine-tuning it is far higher, although the prompts were never trained on. Explain why, using the fact that all positions share the same weights. Would it matter for the model's use as an assistant? Would it matter if you later wanted the same model to keep writing ordinary text?

### Exercise 4 (question): packing conversations

To save compute, SFT libraries often pack several short conversations into one sequence, as notebook 01 packed documents. List what must change so that packing does not change the training task: think about the attention mask (Exercise 1 of notebook 01), the loss mask at the boundary between two conversations, and how the loss is averaged.

## Common errors and debugging

- **Template mismatch.** Training with one template and prompting with another, or forgetting the generation prompt `<|assistant|>`, changes the tokens the model sees. Render both sides with the same function and compare decoded token sequences.
- **Shifting the ids but not the mask.** The mask must follow the targets: `flags[1:]`, not `flags[:-1]`. Print inputs, targets and mask in three aligned rows, as in section 4.
- **Shifting twice.** Some libraries shift the labels inside the model (the causal language models of Hugging Face Transformers do when you pass `labels`); shifting them again outside trains the model to predict two tokens ahead.
- **Forgetting the closing `<|end|>`.** A model never taught to produce it does not stop.
- **Character offsets for token positions.** Our tokeniser has one token per character, but real tokenisers do not, so find the reply boundary in token space, never by string index.
- **Truncating the reply away.** Check every example for at least one supervised token after truncation.
- **Comparing training losses of differently masked runs.** They average over different tokens; compare held-out metrics instead.

## Summary

- A **chat template** serialises roles, content and end markers into one token sequence; training and inference must use the same template, with the generation prompt at inference.
- **Response-only SFT** keeps the prompt in the inputs and removes it from the loss: inputs `ids[:-1]`, targets `ids[1:]`, mask `flags[1:]`, and a mean over supervised tokens.
- **Padding** gets mask 0; **truncation** can delete the reply, so check that every example keeps a supervised token.
- In this notebook, scoring only replies taught all four rules within 250 steps; scoring every token taught the model to write user requests as well, and left it behind on one rule at the same budget.

## Next

In **notebook 03** you adapt the instruction-tuned model to a new instruction without changing a single one of its weights: you build a **LoRA** adapter, prove that the frozen base is untouched and that merging the adapter gives the same outputs, store the frozen weights in 8 bits, and then turn to preference optimisation with the **DPO** loss. Notebook 03 re-creates this notebook's replies-only model with the same recipe, so you can start it directly.